# 👁️ Capstone C2 · See to act: pretrained eyes, two ways to wire them

**Capstone · stage 2 of 4** &nbsp;|&nbsp; ⏱ 60–90 min &nbsp;|&nbsp; 🖥️ Colab **T4 GPU** recommended

In C1 the policy was handed the block's exact position and angle. Real robots don't get that: they get **camera images**.

Modern robot models rarely learn vision from scratch. They take a **pretrained vision encoder** (SigLIP in π0 and SmolVLA, DINOv2 in many research systems), keep it **frozen**, and train on top. You'll wire the same frozen eyes in the two ways industry argues about:

```
END-TO-END   camera ─► DINOv2 (frozen) ─► feature tokens ─────────────────────────► action expert ─► actions
MODULAR      camera ─► DINOv2 (frozen) ─► pose estimator ─► block x, y, angle ─► C1's policy ─► actions
```

**By the end:** a cached DINOv2 feature file (C3 reuses it) · an end-to-end vision policy · a modular perception → control pipeline · both scored on C1's 50 scenes · a clear, measured answer to *"which wiring works with 144 demonstrations?"*

In [ ]:
#@title 🔧 Step 0 · Run this cell first (click ▶). It loads the tools for this lab. { display-mode: "form" }
import importlib.util, subprocess, sys, os
def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
if importlib.util.find_spec("gym_pusht") is None:
    _pip("gym-pusht==0.1.6", "pymunk>=6.6,<7")          # pymunk 7 removed an API gym-pusht needs
if importlib.util.find_spec("zarr") is None or not __import__("zarr").__version__.startswith("2."):
    _pip("zarr==2.18.7", "numcodecs==0.15.1")
if importlib.util.find_spec("imageio") is None:
    _pip("imageio")

import numpy as np, torch, torch.nn.functional as F

# ---------------------------------------------------------------------------
# Guided-lab helpers. You never need to edit this cell.
#  * ___            : a blank for you to fill in
#  * check(name, x) : checks your answer; if it is blank or wrong, it explains
#                     and hands back a working version so the notebook keeps going
#  * quiz(id)       : a clickable multiple-choice question
#  * playground(...) : sliders that re-run a function when you let go
# ---------------------------------------------------------------------------
import inspect, html as _html
import numpy as np
from IPython.display import display, HTML
import os
try:
    import ipywidgets as widgets
    _WIDGETS = not os.environ.get("GUIDE_NO_WIDGETS")
except Exception:
    _WIDGETS = False

class BlankNotFilled(Exception):
    pass

class _Blank:
    """The ___ placeholder. Any maths with it stops with a friendly message."""
    __array_ufunc__ = None
    def _stop(self, *args, **kwargs):
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _stop
    __truediv__ = __rtruediv__ = __floordiv__ = __rfloordiv__ = _stop
    __pow__ = __rpow__ = __matmul__ = __rmatmul__ = __mod__ = __rmod__ = _stop
    __neg__ = __pos__ = __abs__ = __getitem__ = __call__ = __iter__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __bool__ = __float__ = __int__ = __index__ = _stop
    __array__ = __len__ = _stop
    def __getattr__(self, name):
        if name.startswith('__'):
            raise AttributeError(name)
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    def __repr__(self):
        return "___"

___ = _Blank()
CHALLENGES, QUIZZES = {}, {}
_solved, _quiz_score = {}, {}

_STYLE = {
    "ok":   ("#e8f6ee", "#1b7a4b", "✅"),
    "wait": ("#fff5e0", "#9a5b00", "🧩"),
    "bad":  ("#fdecea", "#b3261e", "❌"),
    "info": ("#eaf1fb", "#245eb5", "💡"),
}

def card(kind, title, body=""):
    bg, fg, icon = _STYLE[kind]
    display(HTML(
        f'<div style="background:{bg};border-left:5px solid {fg};padding:10px 14px;'
        f'border-radius:6px;margin:6px 0;color:#1d2530;font-size:14px;line-height:1.5">'
        f'<b style="color:{fg}">{icon} {title}</b><div>{body}</div></div>'))

def _as_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        return [_as_numpy(v) for v in x]
    return x

def _same(a, b, tol):
    a, b = _as_numpy(a), _as_numpy(b)
    if isinstance(a, list) or isinstance(b, list):
        return isinstance(a, list) and isinstance(b, list) and len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    try:
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    except Exception:
        return a == b
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def _has_blank(obj):
    if isinstance(obj, _Blank):
        return True
    if isinstance(obj, dict):
        return any(_has_blank(v) for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return any(_has_blank(v) for v in obj)
    if callable(obj):
        try:
            return "___" in inspect.getsource(obj)
        except Exception:
            return False
    return False

def check(name, answer):
    """Check a challenge. Returns your answer if it works, otherwise a working reference."""
    ch = CHALLENGES[name]
    ref = ch["reference"]
    title = ch.get("title", name)
    fallback = ("<br><i>For now the notebook will use a working version so every later cell still runs. "
                "Come back, fill it in, and re-run this cell.</i>")
    if _has_blank(answer):
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” is waiting for you", "Hint: " + ch["hint"] + fallback)
        return ref
    try:
        if "test" in ch:
            ok, message = ch["test"](answer)
        elif callable(ref):
            ok, message = True, ""
            for args in ch["cases"]:
                args = args if isinstance(args, tuple) else (args,)
                expected, got = ref(*args), answer(*args)
                if not _same(expected, got, ch.get("tol", 1e-6)):
                    ok = False
                    message = "For a test input your function gave a different result from the expected one."
                    break
        else:
            ok = _same(ref, answer, ch.get("tol", 1e-6))
            message = f"You entered <code>{_html.escape(repr(_as_numpy(answer)))}</code>."
    except BlankNotFilled:
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” still has a blank", "Hint: " + ch["hint"] + fallback)
        return ref
    except Exception as err:
        ok, message = False, f"Running your version raised <code>{_html.escape(type(err).__name__)}: {_html.escape(str(err))}</code>."
    if ok:
        _solved[name] = True
        card("ok", f"Challenge solved: {title}", ch.get("why", ""))
        return answer
    _solved[name] = False
    card("bad", f"Not quite yet: {title}", message + "<br>Hint: " + ch["hint"] + fallback)
    return ref

def quiz(qid):
    q = QUIZZES[qid]
    question = f'<div style="font-size:15px;margin:8px 0 4px"><b>{"🔮 Predict: " if q.get("predict") else "🤔 "}{q["q"]}</b></div>'
    if not _WIDGETS:
        options = "".join(f"<li>{_html.escape(o)}</li>" for o in q["options"])
        display(HTML(question + f"<ol type='A'>{options}</ol><details><summary>Answer</summary>"
                     f"{'ABCDEFG'[q['answer']]}. {q['explain']}</details>"))
        return
    out = widgets.Output()
    buttons = []
    def choose(i):
        def handler(_):
            _quiz_score.setdefault(qid, i == q["answer"])
            for j, b in enumerate(buttons):
                b.button_style = "success" if j == q["answer"] else ("danger" if j == i else "")
            with out:
                out.clear_output()
                if i == q["answer"]:
                    card("ok", "Yes!", q["explain"])
                else:
                    card("bad", "Not this one. Here is the reasoning:", q["explain"])
        return handler
    for i, option in enumerate(q["options"]):
        b = widgets.Button(description=f"{'ABCDEFG'[i]}. {option}", layout=widgets.Layout(width="auto", max_width="100%"))
        b.on_click(choose(i))
        buttons.append(b)
    display(HTML(question), widgets.VBox(buttons), out)

def playground(fn, **controls):
    """controls: name=(min, max, step, default) for sliders, or name=[option, ...] for a dropdown."""
    defaults, sliders = {}, {}
    for name, spec in controls.items():
        if isinstance(spec, list):
            defaults[name] = spec[0]
            if _WIDGETS:
                sliders[name] = widgets.Dropdown(options=spec, value=spec[0], description=name)
        else:
            lo, hi, step, value = spec
            defaults[name] = value
            if _WIDGETS:
                kind = widgets.IntSlider if all(isinstance(v, int) for v in spec) else widgets.FloatSlider
                sliders[name] = kind(min=lo, max=hi, step=step, value=value, description=name,
                                     continuous_update=False, style={"description_width": "initial"},
                                     layout=widgets.Layout(width="420px"))
    if _WIDGETS:
        ui = widgets.VBox(list(sliders.values()))
        out = widgets.interactive_output(fn, sliders)
        display(ui, out)
    else:
        fn(**defaults)

def progress_report():
    solved = sum(_solved.values()); total = len(CHALLENGES)
    right = sum(_quiz_score.values()); asked = len(_quiz_score)
    stars = "⭐" * solved + "☆" * (total - solved)
    body = f"Challenges solved yourself: <b>{solved} / {total}</b> {stars}<br>"
    body += f"Quiz questions right on the first click: <b>{right} / {asked}</b> (of {len(QUIZZES)} in this lab)"
    missing = [CHALLENGES[k].get('title', k) for k in CHALLENGES if not _solved.get(k)]
    if missing:
        body += "<br>Still worth a try: " + ", ".join(missing)
    card("info", "Your progress in this lab", body)

# ---- this lab's challenges and quizzes ----
_M = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); _S = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
CHALLENGES["imagenet_norm"] = dict(title="Speak DINOv2's colour language",
    reference=lambda x, mean, std: (x - mean) / std,
    cases=[(torch.rand(2, 3, 8, 8), _M, _S)],
    hint="Subtract the per-channel mean, then divide by the per-channel standard deviation.",
    why="Pretrained encoders expect inputs normalised exactly as during their training (here, ImageNet statistics). Skipping this is a classic silent bug: features look fine but are subtly wrong.")

CHALLENGES["pool"] = dict(title="Shrink 16×16 patches to a 4×4 grid",
    reference=lambda grid: F.adaptive_avg_pool2d(grid, 4),
    cases=[torch.randn(2, 384, 16, 16)],
    hint="<code>F.adaptive_avg_pool2d(grid, 4)</code> averages each 4×4 block of patches into one token.",
    why="256 tokens × 384 numbers per frame is a lot to store and learn from. A 4×4 grid keeps coarse layout (lab 02) at 1/16 of the size: 16 tokens per frame.")

CHALLENGES["standardise"] = dict(title="Standardise with TRAINING statistics",
    reference=lambda z, mu, sd: (z - mu) / sd,
    cases=[(torch.randn(3, 16, 384), torch.randn(384), torch.rand(384) + 0.5)],
    hint="Subtract the training mean <code>mu</code> and divide by the training standard deviation <code>sd</code>.",
    why="Statistics must come from training episodes only. Computing them on test data leaks information, the vision version of lab 03's split rule.")

CHALLENGES["decode"] = dict(title="Turn the estimator's output back into a pose",
    reference=lambda out: np.stack([from_unit(out[:, 0]), from_unit(out[:, 1]), np.arctan2(out[:, 2], out[:, 3]) % (2 * np.pi)], 1),
    cases=[np.array([[0.0, -0.5, 1.0, 0.0], [0.5, 0.25, 0.0, -1.0]])],
    hint="Positions: <code>from_unit</code>. Angle: the estimator outputs (sin, cos), and <code>np.arctan2(sin, cos)</code> turns them back into radians. Wrap into [0, 2π) with <code>% (2 * np.pi)</code>.",
    why="Predicting (sin, cos) instead of the raw angle avoids the 0 = 2π jump (lab 04). Decoding with <code>arctan2</code> is the standard trick in every pose estimator.")

QUIZZES["frozen"] = dict(q="Why keep DINOv2 <b>frozen</b> instead of fine-tuning it with the policy?",
    options=["Frozen models are more accurate", "With only 144 demonstrations, fine-tuning 21 M vision parameters risks overfitting and forgetting general features, and frozen features can be cached, so training is much cheaper", "PyTorch can't fine-tune vision transformers"],
    answer=1, explain="Small robot datasets plus big encoders is a recipe for overfitting. Freezing keeps the general visual knowledge and makes training fast. π0 and SmolVLA freeze or heavily regularise their vision encoders for the same reasons. The cost: the features were never optimised for pushing Ts.")
QUIZZES["vs_state"] = dict(predict=True, q="Compared with C1's policy (exact block pose), the end-to-end vision policy will score…",
    options=["higher: images contain more information", "lower: it must discover what matters in 12,000+ numbers from just 144 demonstrations", "exactly the same"],
    answer=1, explain="C1's 12 numbers are a perfect summary of what matters. End-to-end, the policy must discover that summary itself from two frames × 16 tokens × 384 features, using only the action targets as a teaching signal. With little data it can latch onto details that happen to predict actions in training but not in new scenes.")
QUIZZES["modular"] = dict(predict=True, q="Same frozen DINOv2 features, same 144 demonstrations. Which wiring will score higher?",
    options=["End-to-end: fewer hand-designed parts means fewer errors", "Modular: a pose estimator gets a dense, precise teaching signal on every frame, and C1's policy is already strong", "They must be identical"],
    answer=1, explain="In our test run the modular pipeline scored far higher than end-to-end. The pose estimator learns from an exact label on every frame, a much easier problem than learning control from features. The catch: it needs pose labels, which came free from the simulator but cost a lot on real robots. At very large data scale, end-to-end models (π0, GR00T, Tesla FSD) increasingly win, which is exactly why this debate is alive in industry.")
print('✅ Setup complete. Scroll down and run the cells in order.')

In [ ]:
#@title 🧰 Capstone toolkit · run me (data, simulator, evaluation, GIFs). Read the notes below; no need to edit. { display-mode: "form" }
import os, json, time, math, copy, hashlib, urllib.request
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import gymnasium as gym, gym_pusht, zarr, imageio
from IPython.display import Image as _GifImage
plt.rcParams.update({"figure.dpi": 110})

FAST_DEV_RUN = os.environ.get("CAPSTONE_FAST") == "1"      # course authors' quick self-test switch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
OUT = Path("capstone_outputs"); OUT.mkdir(exist_ok=True)
print("device:", DEVICE, "· outputs folder:", OUT.resolve())
if DEVICE.type == "cpu":
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU. (CPU works, just slower.)")

# ---------------- data ----------------
DATA_URL = "https://diffusion-policy.cs.columbia.edu/data/training/pusht.zip"
def load_pusht(with_images=False):
    archive = Path("pusht.zip")
    if not archive.exists():
        print("Downloading PushT (31 MB)…"); urllib.request.urlretrieve(DATA_URL, archive)
    root = zarr.open_group(store=zarr.ZipStore(str(archive), mode="r"), mode="r", path="pusht/pusht_cchi_v7_replay.zarr")
    data = dict(state=root["data/state"][:].astype(np.float32), action=root["data/action"][:].astype(np.float32),
                ends=root["meta/episode_ends"][:])
    data["starts"] = np.r_[0, data["ends"][:-1]]
    if with_images:
        imgs = np.empty((len(data["state"]), 96, 96, 3), np.uint8)
        for i in range(0, len(imgs), 2048):
            imgs[i:i + 2048] = root["data/img"][i:i + 2048]
        data["images"] = imgs
    order = np.random.default_rng(42).permutation(len(data["ends"]))         # the SAME split as lab 04
    data["train_eps"], data["val_eps"], data["test_eps"] = order[:144], order[144:175], order[175:]
    return data

# ---------------- simulator ----------------
def make_env(obs_type="state"):
    return gym.make("gym_pusht/PushT-v0", obs_type=obs_type)

def sim_state(env):
    u = env.unwrapped
    return np.array([*u.agent.position, *u.block.position, u.block.angle % (2 * np.pi)], np.float32)

def set_dataset_state(env, s):
    """Put the simulator exactly into a recorded dataset state.
    gym-pusht's own reset_to_state sets the block angle AFTER its position. With modern pymunk that rotates
    the T around its centre of mass and moves it ~90 units away from where the dataset recorded it.
    Setting the angle FIRST reproduces the recorded frames pixel-for-pixel (we verified this on all 206 demos)."""
    u = env.unwrapped
    u.agent.position = list(map(float, s[:2])); u.agent.velocity = (0, 0)
    u.block.angle = float(s[4]); u.block.position = list(map(float, s[2:4]))
    u.block.velocity = (0, 0); u.block.angular_velocity = 0
    u.space.step(u.dt)
    return u.get_obs()

def coverage_of(env):
    return float(env.unwrapped._get_coverage())

def score_from_best_coverage(best):
    return min(best / 0.95, 1.0)

# ---------------- evaluation ----------------
EVAL_SEEDS = list(range(100000, 100050))       # 50 fixed start states, identical for every experiment

def run_episode(choose_chunk, seed, execute=8, obs_type="state", record=False, max_steps=300):
    """choose_chunk(history) -> array of future actions (world units). history = list of past observations."""
    env = make_env(obs_type)
    obs, _ = env.reset(seed=seed)
    history, best, frames, steps = [obs, obs], 0.0, [], 0
    while steps < max_steps:
        chunk = choose_chunk(history[-2:], env)
        for a in chunk[:execute]:
            obs, _, terminated, _, info = env.step(np.asarray(a, np.float64))
            history.append(obs); steps += 1
            best = max(best, info["coverage"])
            if record and steps % 2 == 0:
                frames.append(env.unwrapped.render()[::3, ::3])
            if terminated or steps >= max_steps:
                break
        if terminated:
            break
    env.close()
    return dict(best_coverage=best, score=score_from_best_coverage(best), success=best > 0.95, steps=steps, frames=frames)

def bootstrap_ci(values, repeats=2000, seed=0):
    values = np.asarray(values, float); r = np.random.default_rng(seed)
    means = values[r.integers(0, len(values), (repeats, len(values)))].mean(1)
    return float(values.mean()), float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))

def evaluate(choose_chunk, seeds=EVAL_SEEDS, execute=8, obs_type="state", label="policy", verbose=True):
    t0 = time.time(); rows = [run_episode(choose_chunk, s, execute, obs_type) for s in seeds]
    scores = [r["score"] for r in rows]
    mean, lo, hi = bootstrap_ci(scores)
    result = dict(label=label, score=mean, ci95=[lo, hi], success_rate=float(np.mean([r["success"] for r in rows])),
                  reached_80pct=float(np.mean([r["best_coverage"] > 0.8 for r in rows])), episodes=len(rows),
                  seconds=round(time.time() - t0, 1), per_episode_scores=scores)
    if verbose:
        print(f"{label}: score {mean:.3f} (95% CI {lo:.3f}–{hi:.3f}) · ≥80% coverage in {result['reached_80pct']:.0%} "
              f"· full success {result['success_rate']:.0%} · {len(rows)} episodes in {result['seconds']:.0f}s")
    return result

def save_gif(frames, name):
    path = OUT / name
    imageio.mimsave(path, frames, duration=0.1, loop=0)
    return path

def show_gif(path, width=280):
    display(_GifImage(filename=str(path), width=width))

def save_results(name, obj):
    (OUT / name).write_text(json.dumps(obj, indent=1))
    print("saved", OUT / name)

def human_reference(data, episodes):
    env = make_env("state"); env.reset(seed=0); best = []
    for e in episodes:
        b = 0.0
        for i in range(data["starts"][e], data["ends"][e]):
            set_dataset_state(env, data["state"][i]); b = max(b, coverage_of(env))
        best.append(score_from_best_coverage(b))
    return float(np.mean(best))


def to_unit(x):          # world units 0..512  ->  -1..1
    return x / 256.0 - 1.0
def from_unit(x):        # -1..1  ->  world units 0..512
    return (x + 1.0) * 256.0
def state_features(s):   # (…, 5) -> (…, 6): positions in -1..1, angle as sin & cos
    return np.concatenate([to_unit(s[..., :4]), np.sin(s[..., 4:5]), np.cos(s[..., 4:5])], -1).astype(np.float32)

class ResBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm = nn.LayerNorm(width)
        self.ff = nn.Sequential(nn.Linear(width, 4 * width), nn.SiLU(), nn.Linear(4 * width, width))
    def forward(self, h):
        return h + self.ff(self.norm(h))

class FlowPolicy(nn.Module):
    """Velocity network v(noisy action chunk, time, observation)."""
    def __init__(self, obs_dim=12, horizon=16, act_dim=2, width=512, depth=3):
        super().__init__()
        self.horizon, self.act_dim = horizon, act_dim
        self.inp = nn.Linear(horizon * act_dim + obs_dim + 32, width)
        self.blocks = nn.Sequential(*[ResBlock(width) for _ in range(depth)])
        self.out = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, horizon * act_dim))
        self.register_buffer("freqs", torch.exp(torch.linspace(0, math.log(1000), 16)))
    def forward(self, x_t, t, obs):
        tf = t * self.freqs
        h = self.inp(torch.cat([x_t.flatten(1), obs, torch.sin(tf), torch.cos(tf)], 1))
        return self.out(self.blocks(h)).view(-1, self.horizon, self.act_dim)


def load_or_train_c1_policy(data, steps=12000):
    net = FlowPolicy().to(DEVICE)
    ckpt = OUT / "c1_flow_policy.pt"
    if ckpt.exists():
        net.load_state_dict(torch.load(ckpt, map_location=DEVICE)); print("loaded your C1 policy from", ckpt)
        return net.eval()
    print("c1_flow_policy.pt not found in capstone_outputs/, so training one with C1's recipe (upload yours to skip this)…")
    S, A, st, en = data["state"], data["action"], data["starts"], data["ends"]
    obs, chunks = [], []
    for e in data["train_eps"]:
        for i in range(st[e], en[e]):
            obs.append(np.concatenate([state_features(S[max(i - 1, st[e])]), state_features(S[i])]))
            chunks.append(to_unit(A[np.minimum(np.arange(i, i + 16), en[e] - 1)]))
    O, C = torch.tensor(np.array(obs), device=DEVICE), torch.tensor(np.array(chunks), device=DEVICE)
    ema = copy.deepcopy(net).eval(); opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
    steps = 300 if FAST_DEV_RUN else steps
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=steps, pct_start=0.05)
    for step in range(steps):
        b = torch.randint(0, len(O), (256,), device=DEVICE); x1 = C[b]; x0 = torch.randn_like(x1); t = torch.rand(256, 1, 1, device=DEVICE)
        loss = F.mse_loss(net((1 - t) * x0 + t * x1, t.view(256, 1), O[b]), x1 - x0)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step()
        with torch.no_grad():
            d = min(0.999, (1 + step) / (10 + step))
            for pe, pn in zip(ema.parameters(), net.parameters()):
                pe.mul_(d).add_(pn.detach(), alpha=1 - d)
    torch.save(ema.state_dict(), ckpt)
    return ema

@torch.no_grad()
def sample_state_chunks(net, prev_state, state, n=1, flow_steps=10, horizon=16):
    obs = torch.tensor(np.concatenate([state_features(prev_state), state_features(state)]), device=DEVICE)[None].repeat(n, 1)
    x = torch.randn(n, horizon, 2, device=DEVICE)
    for k in range(flow_steps):
        x = x + net(x, torch.full((n, 1), k / flow_steps, device=DEVICE), obs) / flow_steps
    return from_unit(x.clamp(-1, 1)).cpu().numpy()

---
## 1 · Load the data, images included 📷
The images use about 0.7 GB of RAM.

In [ ]:
data = load_pusht(with_images=True)
S, A_, IMGS, starts, ends = data["state"], data["action"], data["images"], data["starts"], data["ends"]
print("images", IMGS.shape, IMGS.dtype)
c1_file = OUT / "c1_results.json"
C1 = json.loads(c1_file.read_text()) if c1_file.exists() else None
HUMAN = C1["human_reference"] if C1 else human_reference(data, data["test_eps"])
print(f"human reference {HUMAN:.3f}" + (f" · your C1 state policy {C1['flow']['score']:.3f}" if C1 else " · (no c1_results.json found; upload it to compare)"))

## 2 · Meet the eyes: DINOv2 🦖
DINOv2-S/14 (21 M parameters, Meta) was trained with self-supervision on 142 M images, with no labels and no robots. It cuts a 224×224 image into a 16×16 grid of 14-pixel patches and returns one 384-number vector per patch.

### 🧩 Challenge 1 · Speak DINOv2's colour language

In [ ]:
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", trust_repo=True).eval().to(DEVICE)
for p in dino.parameters():
    p.requires_grad_(False)                                    # frozen
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def imagenet_normalise(x, mean, std):
    return ___                   # 🧩 per-channel standardisation

imagenet_normalise = check("imagenet_norm", imagenet_normalise)

<details><summary>🤔 <b>Need a hint?</b></summary>

Mean first, then standard deviation.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return (x - mean) / std                   # 🧩 per-channel standardisation</pre>

</details>

### 🧩 Challenge 2 · Shrink 16×16 patches to a 4×4 grid

In [ ]:
def pool_grid(grid):
    return ___     # 🧩 (B, 384, 16, 16) → (B, 384, 4, 4)

pool_grid = check("pool", pool_grid)

@torch.no_grad()
def dino_patches(images_uint8):
    x = torch.as_tensor(np.asarray(images_uint8), device=DEVICE).permute(0, 3, 1, 2).float() / 255
    x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)             # 96 → 224 pixels
    tokens = dino.forward_features(imagenet_normalise(x, MEAN, STD))["x_norm_patchtokens"]
    return tokens.transpose(1, 2).reshape(len(x), 384, 16, 16)                        # (B, 384, 16, 16)

@torch.no_grad()
def dino_tokens(images_uint8):
    return pool_grid(dino_patches(images_uint8)).flatten(2).transpose(1, 2)           # (B, 16, 384)

<details><summary>🤔 <b>Need a hint?</b></summary>

Adaptive pooling to an output size of 4.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return F.adaptive_avg_pool2d(grid, 4)     # 🧩 (B, 384, 16, 16) → (B, 384, 4, 4)</pre>

</details>

### 👀 What does DINOv2 "see"?
A classic DINO visualisation: squeeze each patch's 384 numbers down to 3 with PCA and show them as colours. Nobody told DINOv2 what a T-block is, yet patches of the block, the target and the pusher tend to get distinct colours.

In [ ]:
frames = [starts[e] + 30 for e in data["test_eps"][:4]]
patches = dino_patches(IMGS[frames]).permute(0, 2, 3, 1).reshape(-1, 384).cpu().numpy()     # (4·256, 384)
centred = patches - patches.mean(0)
_, _, vt = np.linalg.svd(centred, full_matrices=False)
rgb = centred @ vt[:3].T
rgb = (rgb - rgb.min(0)) / (rgb.max(0) - rgb.min(0))
fig, axs = plt.subplots(2, 4, figsize=(9, 4.6))
for k, f in enumerate(frames):
    axs[0, k].imshow(IMGS[f]); axs[0, k].axis("off")
    axs[1, k].imshow(rgb[k * 256:(k + 1) * 256].reshape(16, 16, 3)); axs[1, k].axis("off")
axs[0, 0].set_title("camera", loc="left"); axs[1, 0].set_title("DINOv2 patches as colours", loc="left"); plt.show()

## 3 · Cache features for every frame 💾
We run DINOv2 **once** over all 25,650 frames and save the 4×4 tokens (about 315 MB as float16). C3 reuses this file, so keep it.

⏱ About 4–5 minutes on our laptop GPU. The cell prints progress.

In [ ]:
FEAT_FILE = OUT / "dino_4x4.npy"
if FEAT_FILE.exists():
    FEATS = np.load(FEAT_FILE)
    print("loaded cached features", FEATS.shape)
else:
    FEATS = np.zeros((len(IMGS), 16, 384), np.float16); t0 = time.time()
    n = 2048 if FAST_DEV_RUN else len(IMGS)
    for i in range(0, n, 128):
        FEATS[i:i + 128] = dino_tokens(IMGS[i:i + 128]).half().cpu().numpy()
        if i % 5120 == 0: print(f"{i:>6} / {len(IMGS)} frames · {time.time() - t0:.0f}s")
    if not FAST_DEV_RUN:
        np.save(FEAT_FILE, FEATS)
    print(f"done in {time.time() - t0:.0f}s → {FEAT_FILE}")

## 4 · Turn features into observations 🧮
### 🧩 Challenge 3 · Standardise with TRAINING statistics

In [ ]:
quiz("frozen")

In [ ]:
def standardise(z, mu, sd):
    return ___          # 🧩 zero mean, unit spread, using training statistics only

standardise = check("standardise", standardise)

train_frames = np.concatenate([np.arange(starts[e], ends[e]) for e in data["train_eps"]])
val_frames = np.concatenate([np.arange(starts[e], ends[e]) for e in data["val_eps"]])
test_frames = np.concatenate([np.arange(starts[e], ends[e]) for e in data["test_eps"]])
FT = torch.from_numpy(FEATS.astype(np.float32))
MU, SD = FT[train_frames].mean((0, 1)), FT[train_frames].std((0, 1)) + 1e-6
Z = standardise(FT, MU, SD).to(DEVICE)                           # (frames, 16, 384)
POS = torch.from_numpy(to_unit(S[:, :2])).to(DEVICE)             # pusher position only (no block pose!)

H, EXECUTE = 16, 8
cur, prev, chunks = [], [], []
for e in data["train_eps"]:
    for i in range(starts[e], ends[e]):
        cur.append(i); prev.append(max(i - 1, starts[e]))
        chunks.append(to_unit(A_[np.minimum(np.arange(i, i + H), ends[e] - 1)]))
CUR, PREV = torch.tensor(cur, device=DEVICE), torch.tensor(prev, device=DEVICE)
CHUNKS = torch.tensor(np.array(chunks), device=DEVICE)
print("training examples:", len(CUR))

<details><summary>🤔 <b>Need a hint?</b></summary>

Same formula as challenge 1, different statistics.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return (z - mu) / sd          # 🧩 zero mean, unit spread, using training statistics only</pre>

</details>

---
# Wiring A · End-to-end 🔗
Each of the 32 tokens (2 frames × 16) is squeezed from 384 to 64 numbers by a shared layer. Then everything is flattened, concatenated with the pusher position, and fed to C1's flow-matching action expert. A little **dropout** fights overfitting.

In [ ]:
class VisionFlowPolicy(nn.Module):
    def __init__(self, token_dim=64, width=512):
        super().__init__()
        self.token_proj = nn.Sequential(nn.Linear(384, token_dim), nn.LayerNorm(token_dim))
        self.obs_proj = nn.Sequential(nn.Dropout(0.1), nn.Linear(2 * 16 * token_dim + 4, width), nn.SiLU())
        self.expert = FlowPolicy(obs_dim=width, width=width)           # C1's action expert
    def encode(self, z_prev, z_cur, pos_prev, pos_cur):
        tokens = torch.cat([self.token_proj(z_prev).flatten(1), self.token_proj(z_cur).flatten(1), pos_prev, pos_cur], 1)
        return self.obs_proj(tokens)
    def forward(self, x_t, t, obs_embedding):
        return self.expert(x_t, t, obs_embedding)

TRAIN_STEPS = 500 if FAST_DEV_RUN else 20000
torch.manual_seed(0)
vision_policy = VisionFlowPolicy().to(DEVICE); ema = copy.deepcopy(vision_policy).eval()
opt = torch.optim.AdamW(vision_policy.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=TRAIN_STEPS, pct_start=0.05)
t0 = time.time()
for step in range(TRAIN_STEPS):
    b = torch.randint(0, len(CUR), (256,), device=DEVICE)
    emb = vision_policy.encode(Z[PREV[b]], Z[CUR[b]], POS[PREV[b]], POS[CUR[b]])
    x1 = CHUNKS[b]; x0 = torch.randn_like(x1); t = torch.rand(256, 1, 1, device=DEVICE)
    loss = F.mse_loss(vision_policy((1 - t) * x0 + t * x1, t.view(256, 1), emb), x1 - x0)     # C1's flow matching
    opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(vision_policy.parameters(), 1.0); opt.step(); sched.step()
    with torch.no_grad():
        d = min(0.999, (1 + step) / (10 + step))
        for pe, pn in zip(ema.parameters(), vision_policy.parameters()):
            pe.mul_(d).add_(pn.detach(), alpha=1 - d)
    if step % 4000 == 0 or step == TRAIN_STEPS - 1:
        rate = (step + 1) / (time.time() - t0)
        print(f"step {step:>6} · loss {loss.item():.4f} · ~{(TRAIN_STEPS - step) / rate / 60:.1f} min left")
vision_policy = ema
torch.save(dict(policy=vision_policy.state_dict(), mu=MU, sd=SD), OUT / "c2_vision_policy.pt")

### Evaluate: the policy only gets pixels and the pusher position 🎯
At every re-plan we encode the **current** and **previous** camera frames with DINOv2.

In [ ]:
quiz("vs_state")

In [ ]:
def vision_controller(net, flow_steps=10):
    @torch.no_grad()
    def choose(history, env):
        prev_obs, cur_obs = history
        z = standardise(dino_tokens(np.stack([prev_obs["pixels"], cur_obs["pixels"]])).float(), MU.to(DEVICE), SD.to(DEVICE))
        pos = torch.tensor(to_unit(np.stack([prev_obs["agent_pos"], cur_obs["agent_pos"]])), dtype=torch.float32, device=DEVICE)
        emb = net.encode(z[:1], z[1:], pos[:1], pos[1:])
        x = torch.randn(1, H, 2, device=DEVICE)
        for k in range(flow_steps):
            x = x + net(x, torch.full((1, 1), k / flow_steps, device=DEVICE), emb) / flow_steps
        return from_unit(x.clamp(-1, 1))[0].cpu().numpy()[:EXECUTE]
    return choose

seeds = EVAL_SEEDS[:5] if FAST_DEV_RUN else EVAL_SEEDS
end_to_end = evaluate(vision_controller(vision_policy), seeds=seeds, obs_type="pixels_agent_pos", label="A · end-to-end vision policy")
if C1: print(f"C1 state policy: {C1['flow']['score']:.3f}")
print(f"human reference: {HUMAN:.3f}")

---
# Wiring B · Modular: perception, then control 🧩➡️🎮
**Step 1, perception.** A small network reads the 16 DINOv2 tokens of *one* frame and outputs the block's pose as four numbers: x, y (in −1…1) and the angle as (sin, cos). Every training frame has an exact pose label, which is a very rich teaching signal. We add a little noise to the features (augmentation) and keep the checkpoint with the best **validation** error.

**Step 2, control.** Feed *(real pusher position, estimated block pose)* into **C1's policy**, unchanged.

In [ ]:
class PoseEstimator(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_proj = nn.Linear(384, 64)
        self.mlp = nn.Sequential(nn.Dropout(0.2), nn.Linear(16 * 64, 512), nn.SiLU(), nn.Dropout(0.2), nn.Linear(512, 256), nn.SiLU(), nn.Linear(256, 4))
    def forward(self, z):
        return self.mlp(F.silu(self.token_proj(z)).flatten(1))

POSE = torch.from_numpy(np.c_[to_unit(S[:, 2:4]), np.sin(S[:, 4]), np.cos(S[:, 4])]).float().to(DEVICE)

### 🧩 Challenge 4 · Turn the estimator's output back into a pose

In [ ]:
def decode_pose(out):                        # out: (N, 4) = x, y in −1…1, sin(angle), cos(angle)
    x, y = from_unit(out[:, 0]), from_unit(out[:, 1])
    angle = ___     # 🧩 (sin, cos) → radians in [0, 2π)
    return np.stack([x, y, angle], 1)

decode_pose = check("decode", decode_pose)

def pose_errors(net, frame_ids):
    net.eval()
    with torch.no_grad():
        est = decode_pose(net(Z[torch.as_tensor(frame_ids, device=DEVICE)]).cpu().numpy())
    net.train()
    pos = np.linalg.norm(est[:, :2] - S[frame_ids, 2:4], axis=1).mean()
    ang = np.degrees(np.abs(np.arctan2(np.sin(est[:, 2] - S[frame_ids, 4]), np.cos(est[:, 2] - S[frame_ids, 4])))).mean()
    return pos, ang

torch.manual_seed(0)
estimator = PoseEstimator().to(DEVICE)
opt = torch.optim.AdamW(estimator.parameters(), lr=1e-3, weight_decay=1e-3)
TRF = torch.from_numpy(train_frames).to(DEVICE); best = (1e9, None)
for step in range(300 if FAST_DEV_RUN else 8000):
    b = TRF[torch.randint(0, len(TRF), (512,), device=DEVICE)]
    loss = F.mse_loss(estimator(Z[b] + 0.2 * torch.randn_like(Z[b])), POSE[b])          # noisy features = augmentation
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 1000 == 0:
        vp, va = pose_errors(estimator, val_frames)
        if vp < best[0]: best = (vp, copy.deepcopy(estimator.state_dict()))
        print(f"step {step:>5} · validation error {vp:5.1f} units, {va:4.1f}°")
estimator.load_state_dict(best[1]); estimator.eval()
tp, ta = pose_errors(estimator, test_frames)
print(f"TEST pose error: {tp:.1f} units (the T is ~120 units long) · {ta:.1f}°")

<details><summary>🤔 <b>Need a hint?</b></summary>

<code>np.arctan2(sin, cos)</code>, then wrap with <code>% (2 * np.pi)</code>.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>angle = np.arctan2(out[:, 2], out[:, 3]) % (2 * np.pi)     # 🧩 (sin, cos) → radians in [0, 2π)</pre>

</details>

In [ ]:
quiz("modular")

In [ ]:
c1_policy = load_or_train_c1_policy(data)

def modular_controller(flow_steps=10):
    @torch.no_grad()
    def estimate(obs):
        z = standardise(dino_tokens(obs["pixels"][None]).float(), MU.to(DEVICE), SD.to(DEVICE))
        pose = decode_pose(estimator(z).cpu().numpy())[0]
        return np.array([*obs["agent_pos"], *pose], np.float32)        # (pusher x, y, block x, y, angle)
    def choose(history, env):
        prev_obs, cur_obs = history
        return sample_state_chunks(c1_policy, estimate(prev_obs), estimate(cur_obs), flow_steps=flow_steps)[0][:EXECUTE]
    return choose

modular = evaluate(modular_controller(), seeds=seeds, obs_type="pixels_agent_pos", label="B · modular: pose estimator → C1 policy")

In [ ]:
for tag, result, controller in [("end_to_end", end_to_end, vision_controller(vision_policy)), ("modular", modular, modular_controller())]:
    seed = EVAL_SEEDS[int(np.argmin(result["per_episode_scores"][:12]))]
    ep = run_episode(controller, seed, obs_type="pixels_agent_pos", record=True)
    path = save_gif(ep["frames"], f"c2_{tag}_worst12.gif"); print(f"{tag}: worst of the first 12 scenes (seed {seed}) · score {ep['score']:.2f}"); show_gif(path)

---
## 5 · Compare, save, write it up 📝

In [ ]:
rows = [("human demos", HUMAN, None)]
if C1: rows.append(("C1 · true state", C1["flow"]["score"], C1["flow"]["ci95"]))
rows += [("C2 · A end-to-end", end_to_end["score"], end_to_end["ci95"]), ("C2 · B modular", modular["score"], modular["ci95"])]
plt.figure(figsize=(6.5, 0.6 + 0.45 * len(rows)))
names = [r[0] for r in rows]; vals = [r[1] for r in rows]
err = [[v - (r[2][0] if r[2] else v) for v, r in zip(vals, rows)], [(r[2][1] if r[2] else v) - v for v, r in zip(vals, rows)]]
plt.barh(names, vals, xerr=err, capsize=4, color=["#999", "#3b8ea5", "#f2a65a", "#2a9d8f"][:len(rows)]); plt.xlim(0, 1)
plt.gca().invert_yaxis(); plt.xlabel("score on the same 50 scenes (95% CI)"); plt.show()

diff = np.array(modular["per_episode_scores"]) - np.array(end_to_end["per_episode_scores"])
m, lo, hi = bootstrap_ci(diff)
print(f"modular − end-to-end, paired over the same scenes: {m:+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")
end_to_end["label"], modular["label"] = "C2 vision · end-to-end", "C2 vision · modular"
save_results("c2_results.json", dict(stage="C2", human_reference=HUMAN, policy=end_to_end, modular=modular,
                                     pose_error_test=dict(units=float(tp), degrees=float(ta)), train_steps=TRAIN_STEPS))
print("\n📄 Draft résumé bullet:")
print(f"  • Compared end-to-end and modular visuomotor control on frozen DINOv2 features with 144 demonstrations:"
      f" a pose-estimation bottleneck ({tp:.0f}-unit, {ta:.0f}° error) feeding a flow-matching policy scored {modular['score']:.2f}"
      f" vs {end_to_end['score']:.2f} end-to-end on 50 fixed PushT scenes.")

### ✅ Stage checklist
- [ ] `capstone_outputs/dino_4x4.npy` saved (C3 needs it), plus `c2_results.json` and the GIFs
- [ ] A paragraph: *why did the modular wiring win here, and in what situation would you bet on end-to-end instead?* Think about label cost, data scale, and information the pose throws away.

### 🚀 Stretch ideas
* **Fix end-to-end:** stronger feature dropout (0.5), feature noise, fewer steps chosen by validation loss. How much of the gap closes?
* **Spatial detail:** replace the 4×4 grid with the global average of all tokens. What happens to pose error and to each wiring's score?
* **Hybrid:** feed the *estimated pose and* the DINOv2 tokens to the action expert, training the estimator with an auxiliary pose loss. This is how many production stacks combine the two.

**Next:** `C3_Latent_World_Model_PushT.ipynb`. Teach the robot to *imagine* in this same DINOv2 feature space.

In [ ]:
progress_report()